## Pandas-Praxisprojekt: Aktuariat Pricing bei ERGO

Dieses Jupiter Notebook dient einer Übersicht von Pandas- und Numpyfunktionen zur Analyse eines Versicherungsbestands im Bereich Pricing. Der Datensatz und die Aufgabenstellungen sind mit Chatgpt generiert worden.

In [100]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/EikeSuemper/Meine-Lernreise-Aktuariat-QuantFinance/refs/heads/main/04_Datens%C3%A4tze/uebungsdatensatz_gewerbeversicherung.csv", sep = ";", decimal = ",")

### Aufgabe 1: Auffällige Verträge identifizieren

Finde alle Verträge, bei denen:
- Risikoklasse Hoch
- Prämie über 10.000 €
- mindestens ein Schaden
- Schadenquote über 50 %

Zeige nur:
- Vertragsnummer
- Branche
- Region
- Prämie
- Schadenaufwand
- Schadenquote

Zunächst nutze ich die .query() Funktion zum Filtern der Verträge nach obigen Bedingungen.

In [101]:
auffaellige_vertraege = df.query(" (risikoklasse == 'Hoch') and (praemie_eur > 10000) and (anzahl_schaeden >= 1) and ( schadenquote > 0.5 ) ")

Als nächstes möchte ich nur die Spalten Vertragsnummer, Branche, Region, Prämie, Schadenaufwand und Schadenquote anzeigen lassen. Anschließend überschreibe ich auffaellige_vertraege mit der aktuellen Datei.

In [102]:
auffaellige_vertraege = auffaellige_vertraege.loc[:, ["vertrags_id", "region", "praemie_eur", "schadenaufwand_eur", "schadenquote" ]]

In [103]:
auffaellige_vertraege

,vertrags_id,region,praemie_eur,schadenaufwand_eur,schadenquote
63,V100064,Mitte,10792.33,6670.55,0.6181
432,V100433,Ost,23956.12,13961.34,0.5828
718,V100719,Mitte,14549.61,14170.73,0.9740
894,V100895,West,32666.38,20234.87,0.6194
1349,V101350,Sued,25295.32,15198.65,0.6008


### Aufgabe 2: Branchenanalyse mit query()

Finde alle Verträge aus den Branchen Bau, Produktion und Logistik, bei denen:
- der Umsatz über 500.000 € liegt,
- mindestens zwei Schäden aufgetreten sind,
- die Risikoklasse nicht niedrig ist.

Zeige Vertragsnummer, Branche, Risikoklasse, Umsatz und Schadenanzahl. Sortiere nach Umsatz absteigend und zeige die zehn größten Unternehmen.

In [104]:
branchenanalyse = df.query(" (branche in ['Bau', 'Produktion', 'Logistik']) and (anzahl_schaeden >= 2) and (jahresumsatz_eur > 500000) and (risikoklasse != 'Niedrig' ) ").loc[:, ["branche", "risikoklasse", "jahresumsatz_eur", "anzahl_schaeden"]]

Nun sortiere ich die Tabelle nach Umsatz absteigend:

In [105]:
branchenanalyse.sort_values( by = "jahresumsatz_eur", ascending = False) 

,branche,risikoklasse,jahresumsatz_eur,anzahl_schaeden
280,Bau,Mittel,2344786,2
524,Bau,Mittel,1563900,2
1112,Produktion,Mittel,1509989,2
1484,Logistik,Hoch,1320804,2
731,Bau,Hoch,1242174,4
534,Bau,Mittel,1193980,2
416,Bau,Mittel,1013300,2
1031,Logistik,Mittel,984781,2
868,Bau,Mittel,965522,2
1316,Produktion,Hoch,748817,4


### Aufgabe 3: Daten verändern

Erhöhe bei allen Verträgen mit hoher Risikoklasse den Selbstbehalt auf 2.500 €. Speichere anschließend alle betroffenen Verträge in einem neuen DataFrame df_hoch. Zeige Vertragsnummer, Risikoklasse und Selbstbehalt.

In [106]:
df_hoch = df.loc[ df["risikoklasse"] == "Hoch",:] # Dataframe nach Risikoklasse Hoch filtern

In [107]:
df_hoch.loc[:,"selbstbehalt_eur" ] = 2500 # Selbstbehalt durch 2500 Euro ersetzen, wenn Risiko hoch ist

In [108]:
df_hoch.loc[:, ["vertrags_id", "risikoklasse", "selbstbehalt_eur"]] # Ausgewählte Spalten anzeigen

,vertrags_id,risikoklasse,selbstbehalt_eur
11,V100012,Hoch,2500
16,V100017,Hoch,2500
27,V100028,Hoch,2500
34,V100035,Hoch,2500
41,V100042,Hoch,2500
...,...,...,...
1479,V101480,Hoch,2500
1484,V101485,Hoch,2500
1491,V101492,Hoch,2500
1492,V101493,Hoch,2500


### Aufgabe 4: Neue Kategorie erstellen

Erstelle eine neue Spalte "schadenstatus" mit folgenden Attributen:

|Anzahl Schäden| Kategorie|
|-|-|
|0          |	Kein Schaden |
|1          |	Einzelschaden |
|Mehr als 1 |    Mehrere Schäden |

Zeige anschließend alle Verträge mit mehreren Schäden, sortiert nach Schadenaufwand absteigend.